In [ ]:
import re
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score

# =========================
# Name and ID
# =========================
NAME = "Daniel_Tesfai_Kebede"
MATRIKELNUMMER = "1716694"
EVALUATE = True

# =========================
# PATHS
# =========================
TRAIN_PATH = "train.csv"
HOLDBACK_PATH = "holdback_no_targets.csv"
OUT_PATH = f"predictions_{NAME}_{MATRIKELNUMMER}.csv"

# -------------------------
# Cleaning
# -------------------------
def clean_text(s: str) -> str:
    if pd.isna(s):
        return ""
    s = str(s).replace("\u200b", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s

# -------------------------
# Load train data
# -------------------------
train = pd.read_csv(TRAIN_PATH, sep=",")
train = train.dropna(subset=["samples", "targets"]).reset_index(drop=True)
train["samples"] = train["samples"].map(clean_text)

X_all = train["samples"].values
y_all = train["targets"].astype(int).values

# -------------------------
# Model
# -------------------------
tfidf = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3, 5),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)
clf = LinearSVC(C=1.0)

# ============================================================
# 1) EVALUATION MODE F1
# ============================================================
if EVALUATE:
    scores = []

    for seed in range(100):
        # stratified 5% split
        test_parts = []
        for label, grp in train.groupby("targets"):
            n = max(1, int(round(len(grp) * 0.05)))
            test_parts.append(grp.sample(n=n, random_state=seed))

        test = pd.concat(test_parts)
        tr = train.drop(test.index)

        X_tr = tfidf.fit_transform(tr["samples"])
        y_tr = tr["targets"].values
        X_te = tfidf.transform(test["samples"])
        y_te = test["targets"].values

        clf.fit(X_tr, y_tr)
        pred = clf.predict(X_te)

        scores.append(f1_score(y_te, pred, average="macro"))

    scores = np.array(scores)
    print("=== LOCAL EVALUATION ===")
    print(f"Average F1 after 100 runs: {scores.mean():.6f}")
    print(f"Std dev: {scores.std():.6f}")
    print(f"Min / Max: {scores.min():.6f} / {scores.max():.6f}")
    print("========================\n")

# ============================================================
# 2) SUBMISSION
# ============================================================

# Train on FULL train.csv
X_vec = tfidf.fit_transform(X_all)
clf.fit(X_vec, y_all)

# Load holdback
hold = pd.read_csv(HOLDBACK_PATH, sep=",")
hold = hold.dropna(subset=["id", "samples"]).reset_index(drop=True)
hold["samples"] = hold["samples"].map(clean_text)

# Predict
pred = clf.predict(tfidf.transform(hold["samples"].values))

# Export EXACT checker format
submission = pd.DataFrame({
    "id": hold["id"].astype(int),
    "targets": pred.astype(int)
})

submission.to_csv(OUT_PATH, sep=",", index=False)

print("Submission written:", OUT_PATH)
print(submission.head())

=== LOCAL EVALUATION ===
Average F1 after 100 runs: 0.611702
Std dev: 0.063742
Min / Max: 0.425456 / 0.796771

Submission written: /Users/hd/Desktop/Machine Learning/ML-Excercises/predictions_Daniel_Tesfai_Kebede_1716694.csv
    id  targets
0  433        2
1  453        2
2  535        1
3  679        0
4  199        2
